# Exercise 2 — Geometry and the browser Viewer

**Goal:** see the difference between an IFC object's information and its shape. Allow 20–25 minutes. This notebook uses `compas_ifc` for geometry and **COMPAS Notebook Viewer** for an interactive view *inside Binder*.

The desktop `compas_viewer` used by `model.show()` in upstream scripts opens a native window, so it is not the Viewer for this hosted exercise. Binder includes `compas_notebook` for the browser view.

In [ ]:
from pathlib import Path
from compas_ifc.bim import BuildingInformationModel

IFC_PATH = Path("Duplex_A_20110907.ifc")  # CHANGE to sample_building.ifc to compare
if not IFC_PATH.is_file():
    raise FileNotFoundError(f"Missing {IFC_PATH}")
model = BuildingInformationModel(str(IFC_PATH))  # geometry is loaded for this exercise
walls = model.get_elements_by_type("IfcWall")
if not walls:
    raise ValueError("This exercise needs a wall. Start with the bundled sample.")
wall = walls[0]
print(wall.name, wall.ifc_type, wall.global_id)

## 1. Inspect shape and measurements

The same object has a class, name, and GlobalId **and** geometric data. The geometry object's Python class describes its representation. `volume` and `surface_area` are computed from geometry. Check the model's units before interpreting the numbers: the Duplex file declares metres, so its volume is in m³ and area in m². The small sample instead uses millimetres.

In [ ]:
print("Representation:", type(wall.geometry).__name__ if wall.geometry else "none")
print("Volume:", wall.volume)
print("Surface area:", wall.surface_area)
print("GlobalId still:", wall.global_id)

## 2. Make a mesh for display

The surface mesh is a collection of vertices and faces used for viewing. `compute_modeltransformation()` places the local mesh in model coordinates. It does not change the original IFC file.

In [ ]:
wall_mesh = wall.compute_surface_mesh().transformed(wall.compute_modeltransformation())
print("Wall mesh:", len(list(wall_mesh.vertices())), "vertices and", len(list(wall_mesh.faces())), "faces")

## 3. View it in Binder

Run the next cell and drag in the Viewer to rotate. Scroll to zoom. This is a notebook widget; it may take a few seconds to appear. The IFC object's name and properties are still read in Python, while the Viewer displays its mesh.

In [ ]:
from compas_notebook.viewer import Viewer

viewer = Viewer()
viewer.scene.add(wall_mesh)
viewer.show()

## Your turn: add the window

The cell below creates a second Viewer with one window and one wall in model coordinates. They may be far apart because they are simply the first objects of each class. Preview only a few elements at a time.

In [ ]:
windows = model.get_elements_by_type("IfcWindow")
if windows:
    window = windows[0]
    window_mesh = window.compute_surface_mesh().transformed(window.compute_modeltransformation())
    comparison = Viewer()
    comparison.scene.add(wall_mesh)
    comparison.scene.add(window_mesh)
    comparison.show()
else:
    print("No window is recorded in this model.")

## Think and record

- What information can you learn from the Viewer? What requires reading `name`, `ifc_type`, `global_id`, or `properties` instead?
- Change `IFC_PATH` to the small sample and compare its units and shape. If another model has no wall, choose a bundled model before running these cells.
- A computed volume depends on the file's geometry and units. Does it alone tell you material, load, or structural adequacy? Explain.

[Upstream geometry tutorial](https://compas.dev/compas_ifc/latest/tutorials/01_open_a_model.html).